# Проверка гипотезы: дедупликация теряет финальный результат сценария

**Колонки результатов:**
- `DESC_TEXT_1` (в исходном CSV — `DESC_TEXT.1`) — **результат урегулирования по сценарию**
- `DESC_TEXT` — **результат урегулирования ИПУ**

**Гипотеза:** при дедупликации по ключу (ИНН + дата выявления + номер ФП + тип)
`drop_duplicates(keep='first')` оставляет промежуточный результат,
а не финальный. Например, вместо итогового исхода сценария остаётся
«необходим новый план ИПУ».

**Цель:** подтвердить или опровергнуть гипотезу, оценить масштаб проблемы
и найти колонку для сортировки, чтобы гарантированно оставлять последнюю запись.

**Релевантность:** только для отчёта по результатам сценариев. На отчёт по дефолтам не влияет.

## 0. Конфигурация и загрузка

In [ ]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 100)

DATA_DIR = "/home/jovyan/documents/DMUKP_FP_DEF/data"
CRM_FILE = f"{DATA_DIR}/crm_last_version.csv"

DATE_FROM = pd.Timestamp("2023-01-01")
DATE_TO   = pd.Timestamp("2025-12-31")

SEGMENT_MAP = {
    "ДМСБ (микро)":   "МкБ",
    "ДМСБ (малый)":   "МСБ",
    "ДМСБ (средний)": "МСБ",
    "ДКБ":            "ККБ",
}

ALLOWED_SOURCES = ["Н2.0", "Справочник CRM-системы", "CRM-система"]

DEDUP_KEY = ["inn", "fp_num", "fp_type", "IDENTIFICATION_DATE"]

# Колонки-кандидаты для определения порядка записей
ORDER_CANDIDATES = [
    "END_DATE_SCR_FCT",
    "END_DATE_SCR_PLAN",
    "END_EVENT_DATE_FACT",
    "FIRST_END_DATE_EVENT",
    "NEW_PLAN_END_DATE_EVT",
    "DATE_END_FP_SFP",
    "ROW_ID",
]


def normalize_inn(s):
    if pd.isna(s):
        return None
    s = str(s).strip()
    if s.endswith(".0"):
        s = s[:-2]
    s = s.lstrip("0") or "0"
    return s.zfill(10) if len(s) <= 10 else s.zfill(12)


print("OK")

In [ ]:
raw = pd.read_csv(CRM_FILE, dtype=str, low_memory=False)
raw.columns = raw.columns.str.strip()

# В исходном CSV колонка называется DESC_TEXT.1 — переименовываем
if "DESC_TEXT.1" in raw.columns and "DESC_TEXT_1" not in raw.columns:
    raw = raw.rename(columns={"DESC_TEXT.1": "DESC_TEXT_1"})
    print("Переименована колонка DESC_TEXT.1 → DESC_TEXT_1")

print(f"Загружено: {len(raw):,} строк")

if "VAL" in raw.columns:
    raw = raw[raw["VAL"].str.strip().isin(ALLOWED_SOURCES)].copy()
    print(f"После фильтра по источникам: {len(raw):,}")

raw["inn"] = raw["X_INN"].apply(normalize_inn)
raw["dt"] = pd.to_datetime(raw["IDENTIFICATION_DATE"], dayfirst=True, errors="coerce")
raw = raw[(raw["dt"] >= DATE_FROM) & (raw["dt"] <= DATE_TO)].copy()
print(f"После фильтра по периоду: {len(raw):,}")

raw["segment"] = raw["X_AREA_RESP"].str.strip().map(SEGMENT_MAP)
raw = raw[raw["segment"].notna()].copy()

raw["fp_num"] = raw["NUMBER_FP_SFP"].str.strip()
raw["fp_type"] = raw["TYPE_FP"].str.strip()

# DESC_TEXT_1 = результат сценария, DESC_TEXT = результат ИПУ
raw["scenario"] = raw["DESC_TEXT_1"].fillna("").str.strip()
raw["ipu"]      = raw["DESC_TEXT"].fillna("").str.strip()

print(f"Итого для анализа: {len(raw):,} строк, {raw['inn'].nunique():,} ИНН")

## 1. Масштаб проблемы: сколько карточек ФП имеют дубликаты с разными результатами?

- `DESC_TEXT_1` → `scenario` (результат сценария)
- `DESC_TEXT` → `ipu` (результат ИПУ)

In [ ]:
dup_mask = raw.duplicated(subset=DEDUP_KEY, keep=False)
n_total_cards = raw.drop_duplicates(subset=DEDUP_KEY).shape[0]
n_dup_rows = dup_mask.sum()

dup_groups = raw[dup_mask].groupby(DEDUP_KEY)
n_dup_cards = dup_groups.ngroups

print(f"Всего уникальных карточек ФП: {n_total_cards:,}")
print(f"Карточек с дубликатами (>1 строки): {n_dup_cards:,} ({n_dup_cards/n_total_cards*100:.1f}%)")
print(f"Строк в дубликатных группах: {n_dup_rows:,}")

# Сколько дубликатов в группе
grp_sizes = dup_groups.size()
print(f"\nРазмеры дубликатных групп:")
print(grp_sizes.value_counts().sort_index().to_string())

In [ ]:
# Сколько из них имеют РАЗНЫЕ результаты сценария / ИПУ?
sc_nuniq  = dup_groups["scenario"].nunique()
ipu_nuniq = dup_groups["ipu"].nunique()

diff_sc  = sc_nuniq[sc_nuniq > 1]
diff_ipu = ipu_nuniq[ipu_nuniq > 1]

print(f"Карточек с >1 уникальным DESC_TEXT_1 (сценарий): {len(diff_sc):,} из {n_dup_cards:,}")
print(f"Карточек с >1 уникальным DESC_TEXT   (ИПУ):      {len(diff_ipu):,} из {n_dup_cards:,}")

if n_dup_cards > 0:
    print(f"\n→ {len(diff_sc)/n_dup_cards*100:.1f}% дубликатных карточек имеют расхождение в результате сценария")
    print(f"→ {len(diff_ipu)/n_dup_cards*100:.1f}% дубликатных карточек имеют расхождение в результате ИПУ")

## 2. Сравнение keep='first' vs keep='last'

In [ ]:
first_df = raw.drop_duplicates(subset=DEDUP_KEY, keep="first")
last_df  = raw.drop_duplicates(subset=DEDUP_KEY, keep="last")

# Агрегированная разница по результату СЦЕНАРИЯ (DESC_TEXT_1)
first_vc = first_df["scenario"].value_counts()
last_vc  = last_df["scenario"].value_counts()

all_vals = sorted(set(first_vc.index) | set(last_vc.index))
delta = pd.DataFrame({
    "Результат сценария (DESC_TEXT_1)": all_vals,
    "keep=first": [first_vc.get(v, 0) for v in all_vals],
    "keep=last":  [last_vc.get(v, 0) for v in all_vals],
})
delta["Разница"] = delta["keep=last"] - delta["keep=first"]
delta = delta[delta["Разница"] != 0].sort_values("Разница")

if len(delta) > 0:
    print("Результаты СЦЕНАРИЯ (DESC_TEXT_1): разница между keep=first и keep=last")
    print("(+) = чаще при keep=last, (-) = реже при keep=last\n")
    display(delta.style.hide(axis="index").bar(subset=["Разница"], color=["#d65f5f", "#5fba7d"]))
else:
    print("Разницы между keep=first и keep=last для результата сценария нет.")

In [ ]:
# То же для результата ИПУ (DESC_TEXT)
first_vc_ipu = first_df["ipu"].value_counts()
last_vc_ipu  = last_df["ipu"].value_counts()

all_vals_ipu = sorted(set(first_vc_ipu.index) | set(last_vc_ipu.index))
delta_ipu = pd.DataFrame({
    "Результат ИПУ (DESC_TEXT)": all_vals_ipu,
    "keep=first": [first_vc_ipu.get(v, 0) for v in all_vals_ipu],
    "keep=last":  [last_vc_ipu.get(v, 0) for v in all_vals_ipu],
})
delta_ipu["Разница"] = delta_ipu["keep=last"] - delta_ipu["keep=first"]
delta_ipu = delta_ipu[delta_ipu["Разница"] != 0].sort_values("Разница")

if len(delta_ipu) > 0:
    print("Результаты ИПУ (DESC_TEXT): разница между keep=first и keep=last\n")
    display(delta_ipu.style.hide(axis="index").bar(subset=["Разница"], color=["#d65f5f", "#5fba7d"]))
else:
    print("Разницы между keep=first и keep=last для результата ИПУ нет.")

## 3. Примеры расхождений: first vs last для конкретных карточек

In [ ]:
if len(diff_sc) > 0:
    sample_keys = diff_sc.head(30).index.tolist()

    compare = []
    for key in sample_keys:
        inn_v, fp_v, tp_v, dt_v = key
        grp = raw[
            (raw["inn"] == inn_v) &
            (raw["fp_num"] == fp_v) &
            (raw["fp_type"] == tp_v) &
            (raw["IDENTIFICATION_DATE"] == dt_v)
        ].sort_index()
        if len(grp) >= 2:
            compare.append({
                "ИНН": inn_v[:6] + "...",
                "ФП": fp_v,
                "Тип": tp_v,
                "Строк": len(grp),
                "Сценарий (first)": grp.iloc[0]["scenario"][:70] or "[пусто]",
                "Сценарий (last)":  grp.iloc[-1]["scenario"][:70] or "[пусто]",
                "ИПУ (first)": grp.iloc[0]["ipu"][:70] or "[пусто]",
                "ИПУ (last)":  grp.iloc[-1]["ipu"][:70] or "[пусто]",
            })

    if compare:
        print(f"Примеры расхождений ({len(compare)} карточек):")
        display(pd.DataFrame(compare).style.hide(axis="index"))
else:
    print("Расхождений в результатах сценария нет — гипотеза не подтверждается.")

## 4. Полная история одной карточки ФП (детальный пример)

In [ ]:
# Берём первую карточку с расхождением и показываем ВСЕ её записи
if len(diff_sc) > 0:
    key = diff_sc.index[0]
    inn_v, fp_v, tp_v, dt_v = key
    grp = raw[
        (raw["inn"] == inn_v) &
        (raw["fp_num"] == fp_v) &
        (raw["fp_type"] == tp_v) &
        (raw["IDENTIFICATION_DATE"] == dt_v)
    ].sort_index()

    show_cols = [
        "inn", "fp_num", "fp_type", "IDENTIFICATION_DATE",
        "scenario", "ipu", "SCRIPT",
        "END_DATE_SCR_PLAN", "END_DATE_SCR_FCT",
        "FIRST_END_DATE_EVENT", "END_EVENT_DATE_FACT",
        "NEW_PLAN_END_DATE_EVT", "DATE_END_FP_SFP",
        "VAL_1", "ROW_ID",
    ]
    show_cols = [c for c in show_cols if c in grp.columns]

    print(f"Карточка: ИНН={inn_v}, ФП={fp_v}, тип={tp_v}, дата={dt_v}")
    print(f"Записей: {len(grp)}\n")
    display(grp[show_cols].style.hide(axis="index"))
else:
    print("Нет карточек с расхождениями.")

## 5. Кандидаты для сортировки: заполняемость и разброс внутри дубликатных групп

In [ ]:
dup_data = raw[dup_mask].copy()

results = []
for col in ORDER_CANDIDATES:
    if col not in dup_data.columns:
        results.append((col, "НЕТ В ДАННЫХ", "-", "-", "-"))
        continue

    vals = dup_data[col].fillna("").str.strip()
    filled = (vals != "").sum()
    fill_pct = filled / len(dup_data) * 100

    # Сколько групп имеют разные значения этой колонки?
    nuniq = dup_data.groupby(DEDUP_KEY)[col].nunique()
    n_diff = (nuniq > 1).sum()
    pct_diff = n_diff / n_dup_cards * 100 if n_dup_cards > 0 else 0

    results.append((
        col,
        f"{fill_pct:.1f}%",
        f"{filled:,}",
        f"{n_diff:,}",
        f"{pct_diff:.1f}%",
    ))

cand_df = pd.DataFrame(results, columns=[
    "Колонка", "Заполняемость", "Заполнено строк",
    "Групп с разными значениями", "% от дубликатных групп",
])
print("Кандидаты для определения порядка записей (в дубликатных группах):")
display(cand_df.style.hide(axis="index"))

In [ ]:
# Детальнее: для каждого кандидата-даты — распределение разницы (max - min) внутри группы
date_candidates = [c for c in ORDER_CANDIDATES if c != "ROW_ID" and c in dup_data.columns]

for col in date_candidates:
    dup_data[f"_{col}_dt"] = pd.to_datetime(dup_data[col], dayfirst=True, errors="coerce")

    grp_stats = dup_data.groupby(DEDUP_KEY)[f"_{col}_dt"].agg(["min", "max"])
    grp_stats["diff_days"] = (grp_stats["max"] - grp_stats["min"]).dt.days
    has_diff = grp_stats[grp_stats["diff_days"] > 0]["diff_days"]

    if len(has_diff) > 0:
        print(f"\n{col}: {len(has_diff):,} групп с разными датами")
        print(f"  Разница (дни): mean={has_diff.mean():.0f}, "
              f"median={has_diff.median():.0f}, "
              f"min={has_diff.min():.0f}, max={has_diff.max():.0f}")
    else:
        print(f"\n{col}: нет групп с разными датами (не подходит для сортировки)")

## 6. ROW_ID как порядковый номер

In [ ]:
if "ROW_ID" in dup_data.columns:
    dup_data["_row_id_num"] = pd.to_numeric(dup_data["ROW_ID"], errors="coerce")
    filled_rid = dup_data["_row_id_num"].notna().sum()
    print(f"ROW_ID: {filled_rid:,} числовых из {len(dup_data):,}")

    # Проверяем: внутри дубликатных групп ROW_ID различается?
    rid_nuniq = dup_data.groupby(DEDUP_KEY)["_row_id_num"].nunique()
    n_rid_diff = (rid_nuniq > 1).sum()
    print(f"Групп с разными ROW_ID: {n_rid_diff:,} из {n_dup_cards:,}")

    if n_rid_diff > 0:
        # Проверяем гипотезу: запись с MAX ROW_ID имеет финальный результат?
        # Сравниваем: desc из записи с max(ROW_ID) vs desc из last по порядку CSV
        idx_max_rid = dup_data.groupby(DEDUP_KEY)["_row_id_num"].idxmax()
        max_rid_desc = dup_data.loc[idx_max_rid.dropna()].set_index(DEDUP_KEY)["scenario"]

        last_by_csv = raw.drop_duplicates(subset=DEDUP_KEY, keep="last").set_index(DEDUP_KEY)["scenario"]

        common_idx = max_rid_desc.index.intersection(last_by_csv.index)
        match = (max_rid_desc.loc[common_idx] == last_by_csv.loc[common_idx]).sum()
        print(f"\nСовпадение DESC_TEXT: max(ROW_ID) vs keep=last: {match}/{len(common_idx)}")
        print(f"({match/max(len(common_idx),1)*100:.1f}% совпадений)")
else:
    print("Колонка ROW_ID отсутствует.")

## 7. Итоговая рекомендация

In [ ]:
print("="*70)
print("ИТОГ ПРОВЕРКИ")
print("="*70)

if len(diff_sc) == 0:
    print("\n✓ Гипотеза НЕ подтвердилась.")
    print("  Все дубликаты имеют одинаковый результат сценария.")
    print("  Текущая дедупликация корректна.")
else:
    pct = len(diff_sc) / n_total_cards * 100
    print(f"\n⚠ Гипотеза ПОДТВЕРДИЛАСЬ.")
    print(f"  {len(diff_sc):,} карточек ({pct:.2f}% от общего числа) имеют")
    print(f"  разные результаты сценария (DESC_TEXT_1) в дубликатных записях.")
    print(f"\n  При keep=first и keep=last результаты отличаются")
    print(f"  (см. таблицу в секции 2).")
    print(f"\nРекомендация:")
    print(f"  Выбрать колонку для сортировки из секции 5 (с наибольшим")
    print(f"  % групп с разными значениями) и сортировать по убыванию")
    print(f"  перед дедупликацией.")

## 8. Проверка: DESC_TEXT (ИПУ) заполнен только когда сценарий привёл к ИПУ

ИПУ назначается только при определённых негативных исходах сценария.
Если результат сценария (`DESC_TEXT_1`) **не** является одним из «триггерных»,
то результат ИПУ (`DESC_TEXT`) должен быть пустым.

In [ ]:
# Результаты сценария, которые приводят к назначению ИПУ
IPU_TRIGGER_SCENARIOS = [
    "ФП оказывает негативное влияние на исполнение участником кредитной сделки обязательств перед Банком, требуется разработка ПУ",
    "ФП не устранен в рамках сценария, требуется разработка ИПУ",
    "Микро_У ФП не устранен в рамках сценария, требуется разработка ИПУ",
    "10_ККБ/МСБ_ФП не устранен в рамках сценария, оказывает негативное влияние на исполнение участником кредитной сделки обязательств перед Банком, требуется разработка ИПУ",
]

# Работаем с дедуплицированными данными (keep=first — текущее поведение)
df_check = first_df.copy()

df_check["scenario_triggers_ipu"] = df_check["scenario"].isin(IPU_TRIGGER_SCENARIOS)
df_check["ipu_filled"] = df_check["ipu"] != ""

n_total = len(df_check)
n_trigger = df_check["scenario_triggers_ipu"].sum()
n_ipu_filled = df_check["ipu_filled"].sum()

print(f"Всего карточек ФП (после дедупликации keep=first): {n_total:,}")
print(f"Сценарий → ИПУ (триггерный исход): {n_trigger:,} ({n_trigger/n_total*100:.1f}%)")
print(f"DESC_TEXT (ИПУ) заполнен: {n_ipu_filled:,} ({n_ipu_filled/n_total*100:.1f}%)")

# Кросс-таблица
cross = pd.crosstab(
    df_check["scenario_triggers_ipu"].map({True: "Сценарий → ИПУ", False: "Сценарий без ИПУ"}),
    df_check["ipu_filled"].map({True: "ИПУ заполнен", False: "ИПУ пуст"}),
    margins=True, margins_name="Итого"
)
print("\nКросс-таблица:")
display(cross)

# Аномалии: ИПУ заполнен, но сценарий НЕ триггерный
anomalies = df_check[~df_check["scenario_triggers_ipu"] & df_check["ipu_filled"]]
print(f"\n⚠ Аномалии: ИПУ заполнен при НЕтриггерном сценарии: {len(anomalies):,}")

if len(anomalies) > 0:
    print(f"\nРезультаты сценария у аномалий:")
    anom_sc = anomalies["scenario"].value_counts()
    for val, cnt in anom_sc.items():
        print(f"  {cnt:>5,}  {val or '[пусто]'}")

    print(f"\nРезультаты ИПУ у аномалий:")
    anom_ipu = anomalies["ipu"].value_counts()
    for val, cnt in anom_ipu.head(20).items():
        print(f"  {cnt:>5,}  {val}")

In [ ]:
# То же самое, но для keep=last — уменьшается ли количество аномалий?
df_check_last = last_df.copy()
df_check_last["scenario_triggers_ipu"] = df_check_last["scenario"].isin(IPU_TRIGGER_SCENARIOS)
df_check_last["ipu_filled"] = df_check_last["ipu"] != ""

anomalies_last = df_check_last[~df_check_last["scenario_triggers_ipu"] & df_check_last["ipu_filled"]]

print(f"Аномалии (ИПУ заполнен при НЕтриггерном сценарии):")
print(f"  keep=first: {len(anomalies):,}")
print(f"  keep=last:  {len(anomalies_last):,}")
diff_anom = len(anomalies) - len(anomalies_last)
if diff_anom > 0:
    print(f"\n  → keep=last уменьшает аномалии на {diff_anom:,} записей")
elif diff_anom < 0:
    print(f"\n  → keep=last увеличивает аномалии на {-diff_anom:,} записей")
else:
    print(f"\n  → Разницы нет")

# Кросс-таблица для keep=last
cross_last = pd.crosstab(
    df_check_last["scenario_triggers_ipu"].map({True: "Сценарий → ИПУ", False: "Сценарий без ИПУ"}),
    df_check_last["ipu_filled"].map({True: "ИПУ заполнен", False: "ИПУ пуст"}),
    margins=True, margins_name="Итого"
)
print("\nКросс-таблица (keep=last):")
display(cross_last)